In [2]:
%pwd

'/vscmnt/leuven_icts/_data_leuven/365/vsc36564/physioex'

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


In [5]:
import pandas as pd 
import os
os.chdir( os.path.join( os.environ["VSC_DATA"], "physioex" ) )

datasets = ["shhs", "sleepedf", "mass", "dcsm", "hmc"]

models = ["seqsleepnet", "protoseqsleepnet", "protoseqsleepnet.1", "sleeptransformer", "protosleeptransformer", "protosleeptransformer.1"]
columns = ["test/acc", "test/f1", "test/ck","test/pr","test/rc"]

results_df = []
for model in models:
    for dataset in datasets:

        results_path = f"articles/protosleepnet/models/debug/{model}/{dataset}/EEG-EOG-EMG/results.csv"
        robustness_path = f"articles/protosleepnet/models/debug/{model}/{dataset}/EEG-EOG-EMG/robustness.csv"

        res_df = pd.read_csv(results_path)
        if model == "sleeptransformer" and dataset == "shhs":
            res_df = res_df[ [ col.replace("/", "_") for col in columns] ] 
            # rename columns to remove slashes to bring them back with "/"
            res_df.columns = [col.replace("_", "/") for col in res_df.columns]

        else:
            res_df = res_df[columns]
    
        res_df["model"] = model
        res_df["dataset"] = dataset

        if dataset == "shhs":
            res_df["mode"] = "train"
        else:
            res_df["mode"] = "finetune"

        try:
            robustness_df = pd.read_csv(robustness_path)
        except FileNotFoundError:
            print(f"Robustness file not found for {model} on {dataset}. Skipping...")
            results_df.append(res_df)
            continue

        robustness_df = robustness_df[columns]
        # take the average values of the columns
        robustness_df = robustness_df.mean().to_frame().T
        robustness_df["model"] = model
        robustness_df["dataset"] = dataset
        robustness_df["mode"] = "finetune"

        # rename the columns in "columns" to R-<column_name>
        robustness_df = robustness_df.rename(columns={col: f"R-{col}" for col in columns})

        # join the two dataframes over the model, dataset, mode columns
        res_df = res_df.merge(
            robustness_df, 
            on=["model", "dataset", "mode"], 
        )

        results_df.append(res_df)

results_df = pd.concat(results_df, ignore_index=True)

results_df = results_df.rename(
    columns={
        "test/acc": "Acc",
        "test/f1": "F1",
        "test/ck": "Ck",
        "test/pr": "Pr",
        "test/rc": "Rc",
        "R-test/acc": "R-Acc",
        "R-test/f1": "R-F1",
        "R-test/ck": "R-Ck",
        "R-test/pr": "R-Pr",
        "R-test/rc": "R-Rc",
    }
)

# set index to model, dataset, mode
results_df = results_df.set_index(["model", "dataset", "mode"])
results_df = results_df.fillna("-")
results_df = results_df.applymap(lambda x: f"{100*x:.2f}\%" if isinstance(x, float) else x)

# convert float columns to percentage

df_to_export = results_df.copy()

# Esporta in LaTeX ( con multirow e senza crules )

latex_code = df_to_export.to_latex(
    multirow=True,
)
# Salva il codice LaTeX in un file
with open("articles/protosleepnet/results/results_table.tex", "w") as f:
    f.write(latex_code)

results_df.to_csv("articles/protosleepnet/results/results_table.csv")

results_df.head(20)


<>:79: SyntaxWarning: invalid escape sequence '\%'
<>:79: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_4031009/2833467819.py:79: SyntaxWarning: invalid escape sequence '\%'
  results_df = results_df.applymap(lambda x: f"{100*x:.2f}\%" if isinstance(x, float) else x)


Robustness file not found for seqsleepnet on shhs. Skipping...
Robustness file not found for protoseqsleepnet on shhs. Skipping...
Robustness file not found for protoseqsleepnet.1 on shhs. Skipping...
Robustness file not found for sleeptransformer on shhs. Skipping...
Robustness file not found for protosleeptransformer on shhs. Skipping...
Robustness file not found for protosleeptransformer.1 on shhs. Skipping...


/tmp/ipykernel_4031009/2833467819.py:79: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  results_df = results_df.applymap(lambda x: f"{100*x:.2f}\%" if isinstance(x, float) else x)


Acc       F1       Ck       Pr  \
model              dataset  mode                                           
seqsleepnet        shhs     train     87.79\%  87.56\%  81.80\%  89.15\%   
                   sleepedf finetune  81.77\%  80.88\%  73.92\%  83.80\%   
                   mass     finetune  86.56\%  85.97\%  79.95\%  87.10\%   
                   dcsm     finetune  89.63\%  88.80\%  81.65\%  89.84\%   
                   hmc      finetune  74.06\%  72.35\%  64.71\%  75.85\%   
protoseqsleepnet   shhs     train     87.84\%  87.59\%  81.84\%  89.10\%   
                   sleepedf finetune  83.24\%  82.25\%  75.87\%  84.53\%   
                   mass     finetune  86.98\%  85.99\%  80.46\%  87.33\%   
                   dcsm     finetune  90.02\%  89.40\%  82.47\%  90.53\%   
                   hmc      finetune  77.34\%  75.67\%  68.50\%  77.77\%   
protoseqsleepnet.1 shhs     train     86.77\%  85.74\%  79.94\%  87.70\%   
                   sleepedf finetune  82.75\%  81.18\%  74.81\%  83.00\%   
                   mass     finetune  84.92\%  83.29\%  77.14\%  85.80\%   
                   dcsm     finetune  89.63\%  88.30\%  81.32\%  89.53\%   
                   hmc      finetune  75.58\%  73.86\%  66.31\%  77.23\%   
sleeptransformer   shhs     train     87.72\%  87.66\%  81.75\%  89.15\%   
                   sleepedf finetune  84.19\%  83.89\%  77.42\%  85.71\%   
                   mass     finetune  93.69\%  93.52\%  90.63\%  93.57\%   
                   dcsm     finetune  90.83\%  90.42\%  84.03\%  91.55\%   
                   hmc      finetune  79.04\%  78.61\%  71.42\%  81.18\%   

                                           Rc    R-Acc     R-F1     R-Ck  \
model              dataset  mode                                           
seqsleepnet        shhs     train     87.79\%        -        -        -   
                   sleepedf finetune  81.77\%  45.37\%  37.68\%  23.99\%   
                   mass     finetune  86.56\%  63.90\%  56.99\%  43.17\%   
                   dcsm     finetune  89.63\%  66.72\%  64.92\%  45.83\%   
                   hmc      finetune  74.06\%  46.65\%  39.59\%  28.85\%   
protoseqsleepnet   shhs     train     87.84\%        -        -        -   
                   sleepedf finetune  83.24\%  59.94\%  54.69\%  41.31\%   
                   mass     finetune  86.98\%  67.07\%  61.01\%  46.51\%   
                   dcsm     finetune  90.02\%  75.60\%  69.53\%  50.92\%   
                   hmc      finetune  77.34\%  49.80\%  43.45\%  34.24\%   
protoseqsleepnet.1 shhs     train     86.77\%        -        -        -   
                   sleepedf finetune  82.75\%  56.49\%  54.62\%  44.15\%   
                   mass     finetune  84.92\%  59.06\%  51.50\%  30.86\%   
                   dcsm     finetune  89.63\%  75.84\%  70.00\%  49.17\%   
                   hmc      finetune  75.58\%  44.64\%  40.74\%  27.80\%   
sleeptransformer   shhs     train     87.72\%        -        -        -   
                   sleepedf finetune  84.19\%  51.81\%  48.10\%  33.18\%   
                   mass     finetune  93.69\%  58.42\%  54.98\%  41.71\%   
                   dcsm     finetune  90.83\%  76.62\%  73.78\%  56.65\%   
                   hmc      finetune  79.04\%  46.73\%  40.68\%  31.68\%   

                                         R-Pr     R-Rc  
model              dataset  mode                        
seqsleepnet        shhs     train           -        -  
                   sleepedf finetune  47.66\%  45.37\%  
                   mass     finetune  58.88\%  63.90\%  
                   dcsm     finetune  72.01\%  66.72\%  
                   hmc      finetune  43.83\%  46.65\%  
protoseqsleepnet   shhs     train           -        -  
                   sleepedf finetune  58.49\%  59.94\%  
                   mass     finetune  63.43\%  67.07\%  
                   dcsm     finetune  70.20\%  75.60\%  
                   hmc      finetune  49.24\%  49.80\%  
protoseqslee

In [10]:
import pandas as pd

# Seleziona le colonne float e le prime 20 righe
float_cols = ["test_acc", "test_mf1", "test_ck", "test_pr", "test_rc"]
df_to_export = results_df.head(20).copy()

# Applica la colorazione tipo heatmap con pandas Styler
styled = df_to_export.style.background_gradient(subset=float_cols, cmap="YlGnBu")

# Esporta in LaTeX (con multirow e colori)
latex_code = styled.to_latex(hrules=True, convert_css=True, multirow_align="t")

# Salva il codice LaTeX in un file
with open("results_table.tex", "w") as f:
    f.write(latex_code)

In [ ]:
df = pd.read_csv("articles/protosleepnet/results/robustness_results.csv")
df = df[df["occlusion_mask"] == "v2"]
df = df.drop(columns=["occlusion_mask", "trial"])
df = df.groupby(by=["dataset", "model"]).mean().reset_index()
# change the model column to all lowercase
df["model"] = df["model"].str.lower()
df.head()
# merge the results on the dataset and model columns

merged = results_df.merge(
    df, on=["dataset", "model"], how="left", suffixes=("_results", "_robustness")
)
# rimpiazza i valori NaN con "-" e i valori float con .2%
merged = merged.fillna("-")
merged = merged.applymap(lambda x: f"{x:.2%}" if isinstance(x, float) else x)

# rimpiazza i % con \%
merged = merged.replace("%", "\\%", regex=True)

merged = merged.set_index(["dataset", "model", "mode"])

# rename columns
# test_acc	test_f1	test_ck	test_pr	test_rc -> Acc    F1	CK	PR	RC
merged = merged.rename(
    columns={
        "test_acc": "Acc",
        "test_f1": "F1",
        "test_ck": "CK",
        "test_pr": "PR",
        "test_rc": "RC",
    }
)

# re-name robustness columns accuracy	precision	recall	f1_score	cohen_kappa -> Acc-R F1-R CK-R PR-R RC-R
merged = merged.rename(
    columns={
        "accuracy": "Acc-R",
        "precision": "PR-R",
        "recall": "RC-R",
        "f1_score": "F1-R",
        "cohen_kappa": "CK-R",
    }
)
# re-order the columns
merged = merged[
    ["Acc", "F1", "CK", "PR", "RC", "Acc-R", "F1-R", "CK-R", "PR-R", "RC-R"]
]

# Convert the DataFrame to LaTeX code

# set the index model, dataset, mode
merged = merged.reset_index()
merged = merged.set_index(["model", "dataset", "mode"])


latex_code = merged.to_latex()

# Salva il codice LaTeX in un file
with open("articles/protosleepnet/results/results_table.tex", "w") as f:
    f.write(latex_code)

merged.head(n=20)